[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/claude-api-certified/notebooks/day-01-setup.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · Introduction & Claude Models
**certified-journeys / claude-api-certified** · SDK setup & first API call

> **Goal for today:** Install the Anthropic SDK, authenticate, and make your first API call — then compare Haiku, Sonnet, and Opus responses side by side.

In [ ]:
%pip install -q anthropic

## Step 1 · Set up your API key

The Anthropic SDK reads `ANTHROPIC_API_KEY` from the environment. In Colab, use `userdata`; locally, export it in your shell or use a `.env` file.

| Approach | When to use |
|---|---|
| `os.environ` | Quick scripts, local dev |
| Colab `userdata` | Notebook-based work |
| `.env` + `python-dotenv` | Shared repos (never commit the key) |

In [ ]:
import os
import anthropic

# In Colab: add your key in Secrets (🔑 icon) with name ANTHROPIC_API_KEY
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass  # Running locally — key already in environment

client = anthropic.Anthropic()
print("Client created. SDK version:", anthropic.__version__)

**What just happened?**
- `anthropic.Anthropic()` reads `ANTHROPIC_API_KEY` automatically — no need to pass it explicitly.
- **The client is stateless:** each `messages.create()` call is independent.
- All API calls go through `client.messages` — there's only one endpoint to learn.

## Step 2 · Your first messages.create() call

The Messages API follows a chat format: a list of `{role, content}` objects. The minimum viable request needs:
- `model` — which Claude to use
- `max_tokens` — upper bound on the response length
- `messages` — at least one user message

In [ ]:
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,
    messages=[
        {"role": "user", "content": "What is the capital of France? Answer in one sentence."}
    ]
)

# The response object has several useful fields
print("Model:", response.model)
print("Stop reason:", response.stop_reason)
print("Input tokens:", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)
print("\nResponse text:")
print(response.content[0].text)

**What just happened?**
- `response.content` is a list — Claude can return multiple content blocks (text, tool use, etc.).
- **`stop_reason: 'end_turn'`** means Claude finished naturally. `'max_tokens'` means it was cut off.
- **Always check `stop_reason`** in production — a truncated response is a silent failure.
- `usage` gives you exact token counts for cost tracking.

## Step 3 · Understand the model tiers

Anthropic offers three model families with different speed/cost/capability tradeoffs:

| Model | Use case | Relative cost |
|---|---|---|
| `claude-haiku-4-5` | Fast, cheap — great for experiments and high-volume tasks | $ |
| `claude-sonnet-4-5` | Balanced — production workhorse for most applications | $$$  |
| `claude-opus-4-5` | Most capable — complex reasoning, highest quality | $$$$ |

> Always check [docs.anthropic.com/en/docs/about-claude/models](https://docs.anthropic.com/en/docs/about-claude/models) for the latest model IDs — they update frequently.

In [ ]:
import time

PROMPT = "Explain what a closure is in Python. Be concise — 2-3 sentences max."

models = [
    ("claude-haiku-4-5",  "Haiku"),
    ("claude-sonnet-4-5", "Sonnet"),
]

results = []
for model_id, label in models:
    start = time.time()
    r = client.messages.create(
        model=model_id,
        max_tokens=128,
        messages=[{"role": "user", "content": PROMPT}]
    )
    elapsed = time.time() - start
    results.append({
        "label": label,
        "model": model_id,
        "text": r.content[0].text,
        "input_tokens": r.usage.input_tokens,
        "output_tokens": r.usage.output_tokens,
        "latency_s": round(elapsed, 2)
    })

for res in results:
    print(f"\n{'='*50}")
    print(f"[{res['label']}] {res['model']}")
    print(f"Latency: {res['latency_s']}s | Tokens in/out: {res['input_tokens']}/{res['output_tokens']}")
    print(f"Response: {res['text']}")

**What just happened?**
- **Haiku is significantly faster** for short outputs — critical for latency-sensitive apps.
- Token counts are identical (same prompt) but latency differs substantially.
- **Start with Haiku during development.** Switch to Sonnet/Opus only when you need better quality.
- `input_tokens` includes your prompt; `output_tokens` is Claude's response.

## Step 4 · Inspect the full response object

The `Message` object returned by the API contains more than just the text. Understanding its structure is essential for production use.

In [ ]:
import json

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=64,
    messages=[{"role": "user", "content": "Say hello."}]
)

# Convert to dict for inspection
response_dict = response.model_dump()
print(json.dumps(response_dict, indent=2, default=str))

**What just happened?**
- `id` is a unique message ID — useful for logging and debugging.
- `type: 'message'` distinguishes this from streaming events.
- **`content` is always a list** — even for simple text responses.
- `role: 'assistant'` on the response mirrors the request `role: 'user'` convention.

## Step 5 · Extract text safely

Never assume `response.content[0].text` exists. Content blocks can be `TextBlock` or `ToolUseBlock` — always check the type.

In [ ]:
def extract_text(response) -> str:
    """Safely extract text from a Claude response."""
    texts = [
        block.text
        for block in response.content
        if block.type == "text"
    ]
    return "\n".join(texts)


def check_completion(response) -> bool:
    """Return True only if Claude finished naturally (not cut off)."""
    return response.stop_reason == "end_turn"


# Test with a real call
r = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=64,
    messages=[{"role": "user", "content": "List three Python data structures."}]
)

text = extract_text(r)
complete = check_completion(r)

print(f"Complete: {complete}")
print(f"Text: {text}")

**What just happened?**
- **Type-checking content blocks** is a must — tool use adds `ToolUseBlock` alongside text.
- `stop_reason == 'end_turn'` is the only reliable signal that output is complete.
- Building these helpers once saves debugging time across every project.

In [ ]:
# Challenge: Build a simple Q&A function
# Your task: implement ask() so the assertions below pass.

def ask(question: str, model: str = "claude-haiku-4-5", max_tokens: int = 256) -> str:
    """Send a question to Claude and return the text response."""
    # Your solution here
    # Hint: use client.messages.create() and extract_text()
    pass


answer = ask("What year was Python created?")
print("Answer:", answer)
assert isinstance(answer, str) and len(answer) > 0, "ask() should return a non-empty string"
print("✓ Challenge passed!")

---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| `anthropic.Anthropic()` | Reads `ANTHROPIC_API_KEY` automatically |
| `messages.create()` | The single endpoint for all text and tool interactions |
| `response.content` | Always a list — check `.type` before accessing `.text` |
| `stop_reason` | `end_turn` = complete; `max_tokens` = truncated |
| Model tiers | Haiku (fast/cheap) → Sonnet (balanced) → Opus (capable) |

> **Tip:** Use claude-haiku-4-5 while learning — it's 20× cheaper than Opus and fast enough for all experiments.

---
## What's next
**Day 2** → API Core Mechanics: system prompts, multi-turn conversations, streaming, and token counting.

Mark Day 1 complete in your [tracker](../index.html).